In [1]:
# 신용등급평가를 위한 Llama 모델 Unsloth 파인튜닝
import pandas as pd
import numpy as np
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.7.0+cu126 with CUDA 1206 (you have 2.7.0+cu118)
    Python  3.12.10 (you have 3.12.9)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# 1. 환경 설정
max_seq_length = 2048  # 시퀀스 최대 길이
dtype = None  # 자동으로 감지
load_in_4bit = True  # 4bit 양자화 사용

In [ ]:
# 2. 모델과 토크나이저 로드
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",  # 또는 다른 Llama 모델
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

C:\Users\201-03\AppData\Local\miniforge3\Lib\site-packages\unsloth_zoo\gradient_checkpointing.py:330: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  GPU_BUFFERS = tuple([torch.empty(2*256*2048, dtype = dtype, device = f"cuda:{i}") for i in range(n_gpus)])


==((====))==  Unsloth 2025.5.7: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 3050. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu118. CUDA: 8.6. CUDA Toolkit: 11.8. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

In [ ]:
# 3. LoRA 어댑터 추가
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# 4. 데이터 전처리 함수
def create_financial_ratios(df):
    """재무비율 파생변수 생성"""
    df_new = df.copy()
    
    # 안전성 지표
    df_new['부채비율'] = df_new['총부채'] / df_new['자본총계'] * 100
    df_new['유동비율'] = df_new.get('유동자산', df_new['총자산'] * 0.3) / df_new.get('유동부채', df_new['총부채'] * 0.6) * 100
    df_new['자기자본비율'] = df_new['자본총계'] / df_new['총자산'] * 100
    
    # 수익성 지표
    df_new['매출액순이익률'] = df_new['당기순이익'] / df_new['매출액'] * 100
    df_new['총자산순이익률'] = df_new['당기순이익'] / df_new['총자산'] * 100
    df_new['자기자본순이익률'] = df_new['당기순이익'] / df_new['자본총계'] * 100
    df_new['영업이익률'] = df_new['영업이익'] / df_new['매출액'] * 100
    
    # 활동성 지표
    df_new['총자산회전율'] = df_new['매출액'] / df_new['총자산']
    df_new['자기자본회전율'] = df_new['매출액'] / df_new['자본총계']
    
    # 성장성 지표 (전년 대비 - 샘플로 10% 성장률 가정)
    df_new['매출액증가율'] = 10.0  # 실제로는 전년 대비 계산
    df_new['순이익증가율'] = 8.0
    
    # 현금흐름 지표
    df_new['영업현금흐름비율'] = df_new['영업활동현금흐름'] / df_new['매출액'] * 100
    df_new['이자보상배수'] = df_new['영업이익'] / (df_new['이자발생부채'] * 0.05 + 1)  # 가정 이자율 5%
    
    return df_new

In [ ]:
def create_training_prompt(row):
    """훈련용 프롬프트 생성"""
    # 재무 데이터를 자연어로 변환
    prompt = f"""다음 기업의 재무정보를 바탕으로 신용등급을 평가해주세요.

재무정보:
- 매출액: {row['매출액']:,.0f}원
- 영업이익: {row['영업이익']:,.0f}원
- 당기순이익: {row['당기순이익']:,.0f}원
- 총자산: {row['총자산']:,.0f}원
- 총부채: {row['총부채']:,.0f}원
- 자본총계: {row['자본총계']:,.0f}원
- 자본금: {row['자본금']:,.0f}원
- 영업활동현금흐름: {row['영업활동현금흐름']:,.0f}원
- 이자발생부채: {row['이자발생부채']:,.0f}원

재무비율:
- 부채비율: {row['부채비율']:.2f}%
- 자기자본비율: {row['자기자본비율']:.2f}%
- 매출액순이익률: {row['매출액순이익률']:.2f}%
- 총자산순이익률: {row['총자산순이익률']:.2f}%
- 자기자본순이익률: {row['자기자본순이익률']:.2f}%
- 영업이익률: {row['영업이익률']:.2f}%
- 총자산회전율: {row['총자산회전율']:.2f}회
- 영업현금흐름비율: {row['영업현금흐름비율']:.2f}%
- 이자보상배수: {row['이자보상배수']:.2f}배

위 재무정보를 종합적으로 분석하여 신용등급을 평가해주세요."""

    response = f"이 기업의 신용등급은 {row['신용등급']}입니다."
    
    return {
        "instruction": prompt,
        "output": response
    }

In [ ]:
# 5. 데이터 로드 및 전처리
def prepare_dataset(csv_file_path):
    """데이터셋 준비"""
    # CSV 파일 로드
    df = pd.read_csv(csv_file_path)
    
    # 결측값 처리
    df = df.fillna(0)
    
    # 파생변수 생성
    df = create_financial_ratios(df)
    
    # 훈련 데이터 생성
    training_data = []
    for _, row in df.iterrows():
        training_data.append(create_training_prompt(row))
    
    return Dataset.from_list(training_data)

In [ ]:
# 6. 프롬프트 포맷팅 함수
def formatting_prompts_func(examples):
    """Alpaca 스타일 프롬프트 포맷팅"""
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""
        texts.append(text)
    return {"text": texts}

In [ ]:
# 7. 훈련 실행
def train_model(dataset_path, output_dir="./credit_rating_model"):
    """모델 훈련 실행"""
    
    # 데이터셋 준비
    dataset = prepare_dataset(dataset_path)
    dataset = dataset.map(formatting_prompts_func, batched=True)
    
    # 훈련 설정
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            max_steps=100,  # 데이터 크기에 따라 조정
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir=output_dir,
            save_steps=50,
        ),
    )
    
    # 훈련 시작
    trainer_stats = trainer.train()
    
    return trainer_stats

In [ ]:
# 8. 모델 저장
def save_model(output_dir="./credit_rating_model"):
    """모델 저장"""
    # LoRA 어댑터만 저장 (용량 절약)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    # GGUF 형식으로도 저장 (배포용)
    model.save_pretrained_gguf(f"{output_dir}_gguf", tokenizer, quantization_method="q4_k_m")

In [ ]:
# 9. 추론 함수
def predict_credit_rating(financial_data):
    """신용등급 예측"""
    FastLanguageModel.for_inference(model)
    
    # 재무비율 계산
    enhanced_data = create_financial_ratios(pd.DataFrame([financial_data])).iloc[0]
    
    # 프롬프트 생성
    prompt_data = create_training_prompt(enhanced_data)
    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{prompt_data['instruction']}

### Response:
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
    # 응답에서 실제 등급 추출
    response_part = response.split("### Response:")[-1].strip()
    return response_part

In [ ]:
# 10. 사용 예시
if __name__ == "__main__":
    # 훈련 데이터 경로 (실제 CSV 파일 경로로 변경)
    dataset_path = "credit_rating_data.csv"
    
    # 모델 훈련
    print("모델 훈련 시작...")
    stats = train_model(dataset_path)
    print(f"훈련 완료: {stats}")
    
    # 모델 저장
    print("모델 저장 중...")
    save_model()
    print("모델 저장 완료")
    #매출액,영업이익,당기순이익,총자산,총부채,자본총계,자본금,영업활동현금흐름,이자발생부채
    #194836.90229201314,20641.697009139138,23337.90447130256,588966.6008997746,494469.64516028087,105404.57455413367,60042.1724553052,21982.71599877708,88257.84838759775
    # 예측 테스트
    test_data = {
        '신용등급': 'AAA',  # 실제 예측시에는 제외
        '매출액': 194836.90229201314,
        '영업이익': 20641.697009139138,
        '당기순이익': 23337.90447130256,
        '총자산': 588966.6008997746,
        '총부채': 494469.64516028087,
        '자본총계': 105404.57455413367,
        '자본금': 60042.1724553052,
        '영업활동현금흐름': 21982.71599877708,
        '이자발생부채': 88257.84838759775
    }
    
    prediction = predict_credit_rating(test_data)
    print(f"예측 결과: {prediction}")